# Introduction
GOAL: To train a model to predict the wild fire risk of properties, using census data as input features and the proximity to fires as a target feature.

It would be cool to add in data regarding local climate.

In [1]:
import pandas as pd
import load_wildfires
import load_census
import load_properties
import gis
import train
import visualize
import sqlalchemy as s

from pathlib import Path
from sql_funcs import SQL

from settings import PATH_DATA

In [2]:
SQL.kill_idle(True)
sql_obj = SQL()


kill_idle: terminated 8 connection(s)


# Load Properties

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

In [3]:
props = load_properties.Properties(sql_obj=sql_obj)
properties = props.get_properties_gpd()
cur_count = properties.shape[0]
desired_count = 300000
if cur_count < desired_count:
    diff = desired_count - cur_count
    print(f"Adding {diff} more properties...")
    props.add_random_properties_geo_first(diff)


300024 properties loaded.


In [4]:
# result = sql_obj.connection.execute(s.text("SELECT property_id, geoid FROM properties LIMIT 5;"))
# for row in result:
#   print(row)
# raise

# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [5]:
census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')

In [6]:
combined_gdf = census.merge_census_info(properties)
combined_gdf.head()


3109 geographies needed, 3109 cached, 0 to fetch.


,geoid,geometry,B01001A_001E,B01001A_002E,B01001A_003E,B01001A_004E,B01001A_005E,B01001A_006E,B01001A_007E,B01001A_008E,...,B25091_014E,B25091_015E,B25091_016E,B25091_017E,B25091_018E,B25091_019E,B25091_020E,B25091_021E,B25091_022E,B25091_023E
0,261490415002009,POINT (867942.018 2145412.284),NaN,NaN,0.008803,0.010403,0.009105,0.006313,0.003162,0.008226,...,0.203235,0.099250,0.049597,0.026433,0.013752,0.010427,0.004565,0.009807,0.019219,0.001860
1,13281,POINT (1115985.419 1378811.131),NaN,NaN,0.005564,0.006968,0.003709,0.003709,0.005855,0.008876,...,0.235130,0.089972,0.052180,0.020829,0.012454,0.010307,0.014172,0.011166,0.030492,0.005154
2,350079506005040,POINT (-782629.75 1571028.689),NaN,NaN,0.004311,0.003350,0.005959,0.004751,0.001401,0.009941,...,0.296049,0.076193,0.059005,0.067214,0.031298,0.022319,0.024115,0.003848,0.022319,0.001539
3,481319505002228,POINT (-267156.75 484976.634),NaN,NaN,0.002626,0.012823,0.007537,0.004570,0.002455,0.002524,...,0.349341,0.090866,0.123352,0.059793,0.033898,0.007062,0.025895,0.002354,0.031544,0.027307
4,131510703251030,POINT (1094251.991 1221884.054),NaN,NaN,0.003519,0.004272,0.005043,0.003630,0.002049,0.004868,...,0.130039,0.043558,0.023427,0.012725,0.008010,0.003850,0.002659,0.001958,0.009234,0.002007


# Load Wildfire GIS Data for 2024

We will use point data from the Visible Infrared Imaging Radiometer Suite (VIIRS). A valid alternative is using burn boundary data. There are a few different data sources we could use, but in the interest of (portfolio) simplicity we'll use just the VIIRS.

N:B: May be a good chance to practice using AWS DB storage and retrieval?

In [7]:
wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)


Loading previously extracted wildfire data from: ['J1V-C2' 'J1V-C2,SV-C2' 'J1V-C2,J2V-C2' 'J1V-C2,J2V-C2,SV-C2' 'J2V-C2'
 'J2V-C2,SV-C2' 'SV-C2']


In [8]:
# Visualize wildfire locations
# wildfire_map = wildfires.visualize_data(save_path=Path("figures/wildfires_map.html"))
# wildfire_map

## Create Targets (Wildfire Proximity Score)

Give Each Property a Wildfire Risk Score based on the proximity to wildfires.

TODO: List/describe the various options given for targets.


In [9]:
proximity_features = gis.calc_all_features_parallel(combined_gdf, wildfires.data, n_jobs = 5)
targets_features = pd.concat([combined_gdf, proximity_features], axis=1)

Computing GIS features for 300024 properties (5 parallel jobs)...
  Total: 4125.1s
Cached GIS features to gis_features_c66fe57483ce.parquet


In [10]:
targets_features.head()

,geoid,geometry,B01001A_001E,B01001A_002E,B01001A_003E,B01001A_004E,B01001A_005E,B01001A_006E,B01001A_007E,B01001A_008E,...,exp_decay_score,fire_count_0_10km,fire_count_10_25km,fire_count_25_50km,fire_count_50_100km,fire_FRP_0_10km,fire_FRP_10_25km,fire_FRP_25_50km,fire_FRP_50_100km,nearest_fire_km
0,261490415002009,POINT (867942.018 2145412.284),NaN,NaN,0.008803,0.010403,0.009105,0.006313,0.003162,0.008226,...,959.939392,0.0,21.0,63.0,252.0,0.0,221.130,3155.4600,4846.065000,23.659050
1,13281,POINT (1115985.419 1378811.131),NaN,NaN,0.005564,0.006968,0.003709,0.003709,0.005855,0.008876,...,1620.652850,0.0,0.0,399.0,945.0,0.0,0.000,5338.0950,15384.017000,28.296960
2,350079506005040,POINT (-782629.75 1571028.689),NaN,NaN,0.004311,0.003350,0.005959,0.004751,0.001401,0.009941,...,948.351637,0.0,63.0,21.0,63.0,0.0,950.385,1958.4600,980.105000,16.102527
3,481319505002228,POINT (-267156.75 484976.634),NaN,NaN,0.002626,0.012823,0.007537,0.004570,0.002455,0.002524,...,3146.590218,0.0,273.0,168.0,567.0,0.0,3924.536,2031.7675,11837.196000,10.956008
4,131510703251030,POINT (1094251.991 1221884.054),NaN,NaN,0.003519,0.004272,0.005043,0.003630,0.002049,0.004868,...,4803.730225,0.0,63.0,588.0,2835.0,0.0,573.300,11278.4875,58265.593977,20.297246


In [11]:
# # Visualize properties colored by wildfire risk, with wildfire locations
# combined_map = visualize.create_combined_map(
#     targets_features,
#     wildfires.data,
#     risk_column="nearest_fire_km",
#     save_path=Path("figures/risk_map.html")
# )
# combined_map

# Machine Learning Considerations
## Scoring Methods

For the float risk score, we can use Mean Squared Error (MSE) or Root Mean Squared Error (RMSE). Since it's quadratic in difference between observations and predictions deviations, MSE strongly penalizes large misses, which would be expensive for the insurance company.

For the risk category counts, they appear to be Poisson distributed, so a Poisson loss-function is appropriate.

For any classification model with the binned risk categories, we want to make large misses costly (i.e. predicting a 1 when the category is a 10), since these would also be very costly to the insurance company. To be honest, MSE will work here as well, since the categories are just 

# Model Machine Learning


NB: A good chance to make use of AWS compute.


### Split Data into Features/Targets

We use `nearest_fire_km` as the target — the distance in kilometres to the nearest wildfire detection. With the full 300k-property dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a better choice, but the small 119-property test set has nearly zero variance in decay score because all properties are ~360 km from the nearest fire.

All other proximity-derived columns are dropped so the model only sees census features as inputs.

In [12]:
TARGET_COL = "nearest_fire_km"

# All proximity features are derived from the same wildfire data — drop them
# so the model only sees census features as inputs.
proximity_cols = [c for c in proximity_features.columns]
drop_cols = ["geometry", "geoid"] + [c for c in proximity_cols if c != TARGET_COL]

# SAVE TO PARQUET FOR AWS
targets_features.drop(columns=drop_cols).to_parquet(PATH_DATA/"model_joined.parquet")

# Preprocessing with adaptive imputation based on missingness analysis
# - MCAR features: SimpleImputer (median) - fast
# - MAR features: IterativeImputer - preserves correlations
X_train, X_test, y_train, y_test, feature_names, pipeline = train.preprocess_with_cache(
    targets_features,
    TARGET_COL,
    drop_cols,
    nan_threshold=0.45,
    corr_threshold=0.85,
    mar_corr_threshold=0.1,  # Features with missingness corr > 0.1 use IterativeImputer
    use_cache=True,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: {y_train.min():.4f} – {y_train.max():.4f}")
print(f"Target std:   {y_train.std():.6f}")
print(f"Target mean:  {y_train.mean():.4f}")

Computing preprocessing (will cache for future runs)...
  Features before filtering: 1134
  After NaN filter: 848
  After correlation filter: 628
  Analyzing missing data patterns...


C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\U


Missingness Analysis Report:
  MCAR features: 0 (will use median imputation)
  MAR features:  42 (will use iterative imputation)

  Missing data summary:
    Features with missing: 42
    Avg missing %: 12.1%
    Max missing %: 43.6%

  Top 5 MAR features (missingness correlates with other features):
    B25031_006E: 16.7% missing, corr=0.691 with B25031_004E
    B25031_003E: 6.9% missing, corr=0.558 with B25031_004E
    B25031_002E: 39.4% missing, corr=0.541 with B25031_004E
    B19326_007E: 0.1% missing, corr=0.532 with B01001G_029E
    B01002D_002E: 32.0% missing, corr=0.501 with B25063_027E
  Missingness analysis completed in 190.3s
  Fitting adaptive imputation pipeline...


C:\Users\jdcf5\anaconda3\envs\GIS\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  Pipeline fitting completed in 183.5s
  Final feature matrix: (240019, 628)
  Preprocessing completed in 589.7s
  Cached preprocessing to preprocess_e1355c22d306.pkl
Train: (240019, 628), Test: (60005, 628)
Target range: 0.0037 – 316.4062
Target std:   30.173349
Target mean:  25.1604


In [13]:
# Diagnostic: check if the target has meaningful variance
cv = y_train.std() / y_train.mean() * 100  # coefficient of variation
print(f"Coefficient of variation: {cv:.4f}%")
if cv < 1.0:
    print(
        f"WARNING: Target has near-zero variance (CV={cv:.4f}%). "
        f"All properties are ~{y_train.mean():.1f} km from the nearest fire. "
        f"Models will appear to have perfect accuracy but are not learning meaningful patterns. "
        f"Scale to 300k properties for geographic diversity."
    )

Coefficient of variation: 119.9241%


### Preprocessing

Drop high-NaN columns, remove correlated features, then impute + scale. All steps are fit on training data only to prevent leakage.

**Adaptive Imputation:** The preprocessing analyzes each feature's missingness mechanism:
- **MCAR** (Missing Completely At Random): Uses `SimpleImputer(median)` - fast, O(n)
- **MAR** (Missing At Random): Uses `IterativeImputer` - preserves feature correlations

**Caching:** Results are cached to `data/cache/preprocess_{hash}.pkl`. Re-runs with unchanged data load from cache in <1s.

In [14]:
# Preprocessing is now handled by train.preprocess_with_cache() above
# which performs:
#   1. Missingness analysis (MCAR vs MAR detection)
#   2. Adaptive imputation (SimpleImputer for MCAR, IterativeImputer for MAR)
#   3. StandardScaler
# All results are cached - re-runs with unchanged data load in <1s

## Train Models


#### RandomForestRegressor


In [15]:
# rfr_search = train.train_random_forest(X_train, y_train, n_iter=20, cv=5)
# print(f"Best RF params: {rfr_search.best_params_}")
#
# rfr_metrics = train.evaluate_model(rfr_search.best_estimator_, X_train, X_test, y_train, y_test)
# print(f"RF Train RMSE: {rfr_metrics['train_rmse']:.8f}")
# print(f"RF Test  RMSE: {rfr_metrics['test_rmse']:.8f}")

#### XGBoost


In [ ]:
xgb_search = train.train_xgboost(X_train, y_train, n_iter=20, cv=5)
print(f"Best XGB params: {xgb_search.best_params_}")
# model_path = Path("Models") / "best_model.pkl"
# best_model = train.load_model(model_path)['model']

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:

xgb_metrics = train.evaluate_model( xgb_search.best_estimator_
                                   X_train, X_test, y_train, y_test)
print(f"XGB Train RMSE: {xgb_metrics['train_rmse']:.8f}")
print(f"XGB Test  RMSE: {xgb_metrics['test_rmse']:.8f}")


#### Extract Feature Weights





In [ ]:
# Pick the better model
# if xgb_metrics["test_rmse"] <= rfr_metrics["test_rmse"]:
best_model = xgb_search.best_estimator_
print("Best model: XGBoost")
# else:
#     best_model = rfr_search.best_estimator_
#     print("Best model: RandomForest")

top_features = train.extract_feature_importance(best_model, feature_names, top_n=10)
print(f"\nTop 10 features:\n{top_features}")

In [ ]:
# Feature importance bar chart
fig_importance = visualize.plot_feature_importance(
    top_features, 
    title="Top 10 Feature Importances",
    save_path=Path("figures/feature_importance.png")
)
fig_importance

In [ ]:
# Actual vs Predicted scatter plot
y_pred = best_model.predict(X_test)

fig_scatter = visualize.plot_actual_vs_predicted(
    y_test.values, 
    y_pred,
    title="Actual vs Predicted (Test Set)",
    xlabel="Actual Distance to Fire (km)",
    ylabel="Predicted Distance to Fire (km)",
    save_path=Path("figures/actual_vs_predicted.png")
)
fig_scatter

In [ ]:
model_path = Path("Models") / "best_model.pkl"
train.save_model(best_model, model_path, pipeline=pipeline, feature_names=feature_names)
print(f"Model saved to {model_path}")

# Conclusion